%md
### Check 1: labels
I count how many transactions are unknown, licit (2), and illicit (1). Unknown means we don't know the answer, so we won't score those.

In [0]:
labels = spark.table("bronze.elliptic_txs_classes")
display(labels.groupBy("class").count())

%md
### Check 2: label names
I give the labels clear names so the results are easier to read. The saved table does not change.


In [0]:
from pyspark.sql.functions import when

labels_named = labels.withColumn("label_name",
    when(labels["class"] == "1", "illicit")
    .when(labels["class"] == "2", "licit")
    .otherwise("unknown"))

display(labels_named.groupBy("label_name").count())

%md
### Check 3: duplicates
I count all rows and all unique transaction IDs. Both numbers are 203,769, so there are no duplicate transactions.

In [0]:
%sql
SELECT COUNT(*) AS total_rows, COUNT(DISTINCT txId) AS unique_ids
FROM bronze.elliptic_txs_features

%md
### Check: missing values
I count rows where the transaction ID or time step is empty. Result 0 means no empty values in these key columns.

In [0]:
%sql
SELECT COUNT(*) AS broken_links
FROM bronze.elliptic_txs_edgelist
WHERE txId1 NOT IN (SELECT txId FROM bronze.elliptic_txs_features)

### Check: broken links (second end)
I count links whose txId2 is not a transaction in the features table. Result 0 means every link points to a real transaction.


In [0]:
%sql
SELECT COUNT(*) AS broken_links
FROM bronze.elliptic_txs_edgelist
WHERE txId2 NOT IN (SELECT txId FROM bronze.elliptic_txs_features)